# MUST-Former Evaluation Notebook

 Comprehensive evaluation of five model variants on the Sen1Floods11 dataset.
 Evaluates on Test split and Bolivia split separately.

In [23]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
import sys
import subprocess
import sys

packages = [
    "transformers==4.40.0",
    "rasterio",
    "albumentations",
    "scikit-learn",
    "seaborn",
    "matplotlib",
    "pandas",
    "numpy",
    "tqdm"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + package.split())

In [2]:
import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from collections import defaultdict
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class PathConfig:
    S1_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S1Hand"
    S2_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S2Hand"
    LABEL_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/LabelHand"
    JRC_WATER = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/JRCWaterHand"

    TEST_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_test_data.csv"
    BOLIVIA_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_bolivia_data.csv"

    METADATA = "/content/drive/MyDrive/Sen1Floods11/data/Sen1Floods11_Metadata.geojson"
    OUTPUT_DIR = "/content/drive/MyDrive/Sen1Floods11/Evaluation_Outputs"

    def create_dirs(self):
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'predictions'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'confusion_matrices'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'heatmaps'), exist_ok=True)

paths = PathConfig()
paths.create_dirs()

MODEL_PATHS = {
    'MUST-Former-SAR': '/content/drive/MyDrive/checkpoints/MUST-Formers /MUST-Former-SAR.pth',
    'MUST-Former-Optical': '/content/drive/MyDrive/checkpoints/MUST-Formers /MUST-Former-Optical.pth',
    'MUST-Former-Projector': '/content/drive/MyDrive/checkpoints/MUST-Formers /MUST-Former-Projector.pth',
    'MUST-Former-CrossAttn': '/content/drive/MyDrive/checkpoints/MUST-Formers /MUST-Former-CrossAttn.pth',
    'MUST-Former-PCA': '/content/drive/MyDrive/checkpoints/MUST-Formers /MUST-Former-PCA.pth'
}

MODEL_CONFIG_MAP = {
    'MUST-Former-SAR': 's1_only',
    'MUST-Former-Optical': 's2_only',
    'MUST-Former-Projector': 'fusion_projector',
    'MUST-Former-CrossAttn': 'fusion_attention',
    'MUST-Former-PCA': 'fusion_pca'
}

In [19]:
def create_file_mapping(base_dir):
    mapping = {}
    if not os.path.exists(base_dir):
        return mapping
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.tif'):
                full_path = os.path.join(root, file)
                mapping[file] = full_path
    return mapping

s1_mapping = create_file_mapping(paths.S1_HAND)
s2_mapping = create_file_mapping(paths.S2_HAND)
label_mapping = create_file_mapping(paths.LABEL_HAND)

def load_and_process_split(split_path, split_name):
    if not os.path.exists(split_path):
        return None
    df = pd.read_csv(split_path)
    s1_col, label_col = df.columns[0], df.columns[1]
    df = df.rename(columns={s1_col: 's1_filename', label_col: 'label'})

    region_mapping = {
        'Bolivia': 'Bolivia', 'Colombia': 'Colombia', 'Ghana': 'Ghana',
        'India': 'India', 'Cambodia': 'Cambodia', 'Mekong': 'Cambodia',
        'Nigeria': 'Nigeria', 'Pakistan': 'Pakistan', 'Paraguay': 'Paraguay',
        'Somalia': 'Somalia', 'Spain': 'Spain', 'Sri-Lanka': 'Sri-Lanka',
        'Sri Lanka': 'Sri-Lanka', 'USA': 'USA'
    }

    def extract_region_from_filename(filename):
        if pd.isna(filename): return 'unknown'
        filename_str = str(filename).strip()
        for prefix, region in region_mapping.items():
            if filename_str.startswith(prefix): return region
        return 'unknown'

    df['region_id'] = df['label'].apply(extract_region_from_filename).astype(str)
    df['s1_filename'] = df['s1_filename'].astype(str)
    df['label'] = df['label'].astype(str)
    print(f"{split_name}: {len(df)} samples")
    return df

test_df = load_and_process_split(paths.TEST_SPLIT, "Test")
bolivia_df = load_and_process_split(paths.BOLIVIA_SPLIT, "Bolivia")

In [13]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

class MultiModalSegFormer(nn.Module):
    def __init__(self, backbone='nvidia/mit-b2', num_classes=2, s1_channels=2,
                 s2_channels=13, include_s1=True, include_s2=True,
                 fusion_type='projector', pretrained=True, dropout=0.1):
        super().__init__()
        self.include_s1 = include_s1
        self.include_s2 = include_s2
        self.fusion_type = fusion_type
        self.s1_channels = s1_channels if include_s1 else 0
        self.s2_channels = s2_channels if include_s2 else 0
        self.extra_channels = 2 if include_s2 else 0

        if include_s1 and include_s2:
            self.total_input_channels = self.s1_channels + self.s2_channels + self.extra_channels
        elif include_s1:
            self.total_input_channels = self.s1_channels
        elif include_s2:
            self.total_input_channels = self.s2_channels + self.extra_channels
        else:
            raise ValueError("At least one of include_s1 or include_s2 must be True")

        if fusion_type == 'projector':
            self._init_projector_fusion()
        elif fusion_type == 'attention':
            self._init_attention_fusion()
        elif fusion_type == 'pca':
            self._init_pca_fusion()
        else:
            raise ValueError(f"Unknown fusion_type: {fusion_type}")

        if pretrained:
            try:
                self.segformer = SegformerForSemanticSegmentation.from_pretrained(
                    backbone, num_labels=num_classes, ignore_mismatched_sizes=True
                )
            except Exception:
                config = SegformerConfig.from_pretrained(backbone)
                config.num_labels = num_classes
                self.segformer = SegformerForSemanticSegmentation(config)
        else:
            config = SegformerConfig.from_pretrained(backbone)
            config.num_labels = num_classes
            self.segformer = SegformerForSemanticSegmentation(config)

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def _init_projector_fusion(self):
        if self.include_s1 and not self.include_s2:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)
        elif self.include_s2 and not self.include_s1:
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)
        else:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, 3, 1), nn.BatchNorm2d(3))
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)

    def _init_attention_fusion(self):
        proj_dim = 128
        num_heads = 4
        spatial_reduction = 16

        if self.include_s1:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU())
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)

        if self.include_s2:
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU())
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)

        self.spatial_reduction = spatial_reduction
        if spatial_reduction > 1:
            self.downsample = nn.Sequential(
                nn.Conv2d(proj_dim, proj_dim, spatial_reduction, stride=spatial_reduction, groups=proj_dim),
                nn.BatchNorm2d(proj_dim)
            )
            nn.init.xavier_uniform_(self.downsample[0].weight, gain=0.02)

        self.fusion_norm1 = nn.LayerNorm(proj_dim)
        self.fusion_norm2 = nn.LayerNorm(proj_dim)
        self.cross_attn_12 = nn.MultiheadAttention(proj_dim, num_heads, dropout=0.0, batch_first=True)
        self.cross_attn_21 = nn.MultiheadAttention(proj_dim, num_heads, dropout=0.0, batch_first=True)

        self.ffn1 = nn.Sequential(
            nn.Linear(proj_dim, int(proj_dim * 1.5)), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(int(proj_dim * 1.5), proj_dim), nn.Dropout(0.1)
        )
        self.ffn2 = nn.Sequential(
            nn.Linear(proj_dim, int(proj_dim * 1.5)), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(int(proj_dim * 1.5), proj_dim), nn.Dropout(0.1)
        )

        for module in [self.ffn1, self.ffn2]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight, gain=0.02)
                    nn.init.zeros_(layer.bias)

        self.norm_ffn1 = nn.LayerNorm(proj_dim)
        self.norm_ffn2 = nn.LayerNorm(proj_dim)

        self.fusion_proj = nn.Sequential(
            nn.Conv2d(proj_dim * 2, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU(),
            nn.Conv2d(proj_dim, 3, 1), nn.BatchNorm2d(3)
        )
        self.single_proj = nn.Sequential(nn.Conv2d(proj_dim, 3, 1), nn.BatchNorm2d(3))

        nn.init.xavier_uniform_(self.fusion_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.fusion_proj[0].bias)
        nn.init.xavier_uniform_(self.fusion_proj[3].weight, gain=0.02)
        nn.init.zeros_(self.fusion_proj[3].bias)
        nn.init.xavier_uniform_(self.single_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.single_proj[0].bias)

    def _init_pca_fusion(self):
        self.pca_proj = nn.Sequential(nn.Conv2d(self.total_input_channels, 3, 1), nn.BatchNorm2d(3))
        nn.init.xavier_uniform_(self.pca_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.pca_proj[0].bias)

    def forward(self, x, metadata=None):
        B, C, H, W = x.shape

        expected = (self.s1_channels + self.s2_channels + self.extra_channels) if \
                   (self.include_s1 and self.include_s2) else \
                   (self.s1_channels or (self.s2_channels + self.extra_channels))
        assert C == expected, f"Channel mismatch: got {C}, expected {expected}"

        if self.fusion_type == 'projector':
            x_fused = self._forward_projector(x)
        elif self.fusion_type == 'attention':
            x_fused = self._forward_attention(x)
        else:
            x_fused = self._forward_pca(x)

        x_fused = self.dropout(x_fused)
        outputs = self.segformer(pixel_values=x_fused)
        logits = outputs.logits
        upsampled = nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return upsampled

    def _forward_projector(self, x):
        if self.include_s1 and not self.include_s2:
            return self.s1_proj(x)
        elif self.include_s2 and not self.include_s1:
            return self.s2_proj(x)
        else:
            s1_out = self.s1_proj(x[:, :self.s1_channels])
            s2_out = self.s2_proj(x[:, self.s1_channels:])
            return s1_out + s2_out

    def _forward_attention(self, x):
        features = []
        if self.include_s1:
            features.append(self.s1_proj(x[:, :self.s1_channels]))
        if self.include_s2:
            s2_start = self.s1_channels if self.include_s1 else 0
            s2_end = s2_start + self.s2_channels + self.extra_channels
            features.append(self.s2_proj(x[:, s2_start:s2_end]))

        if len(features) == 2:
            f1, f2 = features
            B_, C_, H_, W_ = f1.shape

            if getattr(self, 'spatial_reduction', 1) > 1:
                f1d = self.downsample(f1)
                f2d = self.downsample(f2)
                _, _, Hd, Wd = f1d.shape
            else:
                f1d, f2d, Hd, Wd = f1, f2, H_, W_

            f1_tokens = f1d.flatten(2).permute(0, 2, 1)
            f2_tokens = f2d.flatten(2).permute(0, 2, 1)

            f1n = self.fusion_norm1(f1_tokens)
            f2n = self.fusion_norm2(f2_tokens)
            f1n = f1n / (f1n.std(dim=-1, keepdim=True) + 1e-8)
            f2n = f2n / (f2n.std(dim=-1, keepdim=True) + 1e-8)

            attn1, _ = self.cross_attn_12(f1n, f2n, f2n)
            attn1 = attn1 + f1_tokens
            attn1 = attn1 + self.ffn1(self.norm_ffn1(attn1))

            attn2, _ = self.cross_attn_21(f2n, f1n, f1n)
            attn2 = attn2 + f2_tokens
            attn2 = attn2 + self.ffn2(self.norm_ffn2(attn2))

            attn1 = attn1.permute(0, 2, 1).reshape(B_, C_, Hd, Wd)
            attn2 = attn2.permute(0, 2, 1).reshape(B_, C_, Hd, Wd)

            if getattr(self, 'spatial_reduction', 1) > 1:
                attn1 = nn.functional.interpolate(attn1, size=(H_, W_), mode='bilinear', align_corners=False)
                attn2 = nn.functional.interpolate(attn2, size=(H_, W_), mode='bilinear', align_corners=False)

            return self.fusion_proj(torch.cat([attn1, attn2], dim=1))
        else:
            return self.single_proj(features[0])

    def _forward_pca(self, x):
        x_mean = x.mean(dim=1, keepdim=True)
        x_std = x.std(dim=1, keepdim=True) + 1e-8
        return self.pca_proj((x - x_mean) / x_std)


def create_model(model_type='fusion', backbone='nvidia/mit-b2', fusion_type='projector', pretrained=True, num_classes=2):
    include_s1 = model_type in ['fusion', 's1_only', 'sar1']
    include_s2 = model_type in ['fusion', 's2_only', 'sar2']

    return MultiModalSegFormer(
        backbone=backbone,
        num_classes=num_classes,
        s1_channels=2,
        s2_channels=13,
        include_s1=include_s1,
        include_s2=include_s2,
        fusion_type=fusion_type,
        pretrained=pretrained
    )

MODEL_CONFIGS = {
    'fusion_projector': {'model_type': 'fusion', 'fusion_type': 'projector'},
    'fusion_attention': {'model_type': 'fusion', 'fusion_type': 'attention'},
    'fusion_pca': {'model_type': 'fusion', 'fusion_type': 'pca'},
    's1_only': {'model_type': 's1_only', 'fusion_type': 'projector'},
    's2_only': {'model_type': 's2_only', 'fusion_type': 'projector'},
}

def get_model_by_name(name, pretrained=True, num_classes=2, backbone='nvidia/mit-b2'):
    if name not in MODEL_CONFIGS:
        raise ValueError(f"Unknown model name: {name}. Available: {list(MODEL_CONFIGS.keys())}")
    cfg = MODEL_CONFIGS[name]
    return create_model(
        model_type=cfg['model_type'],
        backbone=backbone,
        fusion_type=cfg['fusion_type'],
        pretrained=pretrained,
        num_classes=num_classes
    )

In [5]:
class Sen1Floods11Dataset(Dataset):
    def __init__(self, df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
                 transform=None, include_s1=True, include_s2=True):
        self.df = df.reset_index(drop=True)
        self.s1_mapping = s1_mapping
        self.s2_mapping = s2_mapping
        self.label_mapping = label_mapping
        self.jrc_mapping = jrc_mapping
        self.transform = transform
        self.include_s1 = include_s1
        self.include_s2 = include_s2

    def load_tiff(self, path):
        try:
            with rasterio.open(path) as src:
                data = src.read()
                data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
                return data
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None

    def compute_indices(self, s2_data):
        green = s2_data[2].astype(np.float32)
        red = s2_data[3].astype(np.float32)
        nir = s2_data[7].astype(np.float32)
        eps = 1e-8

        ndvi = (nir - red) / (nir + red + eps)
        ndvi = np.nan_to_num(np.clip(ndvi, -1, 1), nan=0.0)

        ndwi = (green - nir) / (green + nir + eps)
        ndwi = np.nan_to_num(np.clip(ndwi, -1, 1), nan=0.0)

        return ndvi, ndwi

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s1_filename = row.iloc[0]
        label_filename = row.iloc[1]
        s2_filename = label_filename.replace('LabelHand', 'S2Hand')

        s1_path = self.s1_mapping.get(s1_filename)
        s2_path = self.s2_mapping.get(s2_filename)
        label_path = self.label_mapping.get(label_filename)

        s1 = self.load_tiff(s1_path) if s1_path and self.include_s1 else None
        s2 = self.load_tiff(s2_path) if s2_path and self.include_s2 else None
        label = self.load_tiff(label_path)

        if label is None:
            raise ValueError(f"Could not load label for {label_filename}")
        label = label[0].astype(np.int64)

        mask_invalid = ~np.isin(label, [-1, 0, 1, 255])
        if mask_invalid.any():
            label = np.where(mask_invalid, 255, label)
        label = np.where(label == -1, 255, label)

        channels = []

        if self.include_s1 and s1 is not None:
            s1_vv = s1[0].astype(np.float32)
            s1_vh = s1[1].astype(np.float32)

            s1_vv = np.nan_to_num(s1_vv, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vh = np.nan_to_num(s1_vh, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vv = np.clip(s1_vv, -50, 0)
            s1_vh = np.clip(s1_vh, -50, 0)
            s1_vv = (s1_vv + 50) / 50.0
            s1_vh = (s1_vh + 50) / 50.0

            channels.extend([s1_vv, s1_vh])

        if self.include_s2 and s2 is not None:
            s2f = s2.astype(np.float32)
            s2f = np.nan_to_num(s2f, nan=0.0, posinf=10000.0, neginf=0.0)
            s2n = np.clip(s2f / 10000.0, 0, 1)

            for i in range(s2n.shape[0]):
                channels.append(s2n[i])

            ndvi, ndwi = self.compute_indices(s2f)
            channels.extend([ndvi, ndwi])

        if len(channels) == 0:
            raise ValueError("No input channels available")

        image = np.stack(channels, axis=-1)

        if np.isnan(image).any() or np.isinf(image).any():
            image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)

        if self.transform:
            augmented = self.transform(image=image, mask=label)
            image = augmented['image']
            label = augmented['mask']
            label = label.long() if isinstance(label, torch.Tensor) else torch.from_numpy(label).long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            label = torch.from_numpy(label).long()

        if torch.isnan(image).any() or torch.isinf(image).any():
            image = torch.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)
        if label.min() < 0 or label.max() > 255:
            label = torch.clamp(label, 0, 255)

        region_id = str(label_filename).split('_')[0]
        meta = {'region_id': region_id, 'filename': label_filename}

        return image, label, meta

def get_val_transforms():
    return A.Compose([ToTensorV2()])

def create_eval_loader(df, batch_size=4, include_s1=True, include_s2=True):
    dataset = Sen1Floods11Dataset(
        df, s1_mapping, s2_mapping, label_mapping, None,
        get_val_transforms(), include_s1, include_s2
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return loader, dataset

In [6]:
def compute_metrics(all_preds, all_targets):
    mask = all_targets != 255
    preds = all_preds[mask]
    targets = all_targets[mask]

    cm = confusion_matrix(targets, preds, labels=[0, 1])

    intersection = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - intersection
    iou = intersection / (union + 1e-8)

    acc = np.diag(cm).sum() / (cm.sum() + 1e-8)
    precision = cm[1, 1] / (cm[1, 1] + cm[0, 1] + 1e-8)
    recall = cm[1, 1] / (cm[1, 1] + cm[1, 0] + 1e-8)

    return {
        'bg_iou': iou[0],
        'water_iou': iou[1],
        'mean_iou': np.mean(iou),
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'cm': cm
    }

def evaluate_model_with_regions(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    region_preds = defaultdict(list)
    region_targets = defaultdict(list)

    with torch.no_grad():
        for data, target, meta in tqdm(loader, desc="Evaluating"):
            data = data.to(device).float()
            target = target.to(device).long()
            logits = model(data)
            preds = logits.argmax(dim=1)

            p = preds.cpu().numpy().flatten()
            t = target.cpu().numpy().flatten()

            all_preds.extend(p)
            all_targets.extend(t)

            regions = meta['region_id'] if isinstance(meta, dict) else [m['region_id'] for m in meta]
            b_size = data.shape[0]
            pixels_per_image = p.shape[0] // b_size
            for i in range(b_size):
                rid = regions[i]
                start = i * pixels_per_image
                end = (i + 1) * pixels_per_image
                region_preds[rid].extend(p[start:end])
                region_targets[rid].extend(t[start:end])

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    overall_metrics = compute_metrics(all_preds, all_targets)

    region_metrics = {}
    for rid in region_preds:
        rp = np.array(region_preds[rid])
        rt = np.array(region_targets[rid])
        region_metrics[rid] = compute_metrics(rp, rt)

    return overall_metrics, region_metrics

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
results = {}

for model_name, path in MODEL_PATHS.items():
    print(f"Loading {model_name} ")
    cfg_name = MODEL_CONFIG_MAP[model_name]

    include_s1 = cfg_name in ['s1_only', 'fusion_projector', 'fusion_attention', 'fusion_pca']
    include_s2 = cfg_name in ['s2_only', 'fusion_projector', 'fusion_attention', 'fusion_pca']

    model = get_model_by_name(cfg_name, pretrained=False, num_classes=2).to(device)

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    test_loader, _ = create_eval_loader(test_df, include_s1=include_s1, include_s2=include_s2)
    bolivia_loader, _ = create_eval_loader(bolivia_df, include_s1=include_s1, include_s2=include_s2)

    print("Evaluating on Test Split")
    test_metrics, test_region_metrics = evaluate_model_with_regions(model, test_loader, device)

    print("Evaluating on Bolivia Split")
    bolivia_metrics, bolivia_region_metrics = evaluate_model_with_regions(model, bolivia_loader, device)

    results[model_name] = {
        'test': test_metrics,
        'test_regions': test_region_metrics,
        'bolivia': bolivia_metrics,
        'bolivia_regions': bolivia_region_metrics,
        'include_s1': include_s1,
        'include_s2': include_s2
    }

    del model
    torch.cuda.empty_cache()

for model_name, res in results.items():
    print(f"\n=== {model_name} ===")
    for split_name in ['test', 'bolivia']:
        m = res[split_name]
        print(f"[{split_name.upper()}] mIoU: {m['mean_iou']:.4f} | Water IoU: {m['water_iou']:.4f} | BG IoU: {m['bg_iou']:.4f}")
        print(f"  Accuracy: {m['accuracy']:.4f} | Precision: {m['precision']:.4f} | Recall: {m['recall']:.4f}")

        plt.figure(figsize=(6, 5))
        sns.heatmap(m['cm'], annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Background', 'Water'],
                    yticklabels=['Background', 'Water'])
        plt.title(f"{model_name} - {split_name.capitalize()} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(paths.OUTPUT_DIR, 'confusion_matrices', f"{model_name}_{split_name}_cm.png"))
        plt.show()

In [21]:
dataset_full = Sen1Floods11Dataset(
    test_df, s1_mapping, s2_mapping, label_mapping, None,
    get_val_transforms(), include_s1=True, include_s2=True
)

print(" generate prediction samples")
all_preds_dict = {m: [] for m in MODEL_PATHS.keys()}

for model_name, path in tqdm(MODEL_PATHS.items(), desc="Models"):
    cfg_name = MODEL_CONFIG_MAP[model_name]
    include_s1 = cfg_name in ['s1_only', 'fusion_projector', 'fusion_attention', 'fusion_pca']
    include_s2 = cfg_name in ['s2_only', 'fusion_projector', 'fusion_attention', 'fusion_pca']

    model = get_model_by_name(cfg_name, pretrained=False, num_classes=2).to(device)
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()

    preds_list = []
    with torch.no_grad():
        loader = DataLoader(dataset_full, batch_size=4, shuffle=False, num_workers=2)
        for data, target, meta in loader:
            if include_s1 and not include_s2:
                inp = data[:, :2].to(device).float()
            elif include_s2 and not include_s1:
                inp = data[:, 2:].to(device).float()
            else:
                inp = data.to(device).float()

            logits = model(inp)
            preds = logits.argmax(dim=1).cpu().numpy()
            preds_list.extend(preds)

    all_preds_dict[model_name] = np.array(preds_list)
    del model
    torch.cuda.empty_cache()

print("Collecting inputs and ground truths...")
all_inputs = []
all_gts = []
all_meta = []
loader_full = DataLoader(dataset_full, batch_size=4, shuffle=False, num_workers=2)
for data, target, meta in loader_full:
    all_inputs.extend(data.cpu().numpy())
    all_gts.extend(target.cpu().numpy())
    all_meta.extend(meta['filename'] if isinstance(meta, dict) else [m['filename'] for m in meta])

all_inputs = np.array(all_inputs)
all_gts = np.array(all_gts)

num_samples = len(all_inputs)
samples_to_show_idx = np.linspace(0, num_samples-1, 5, dtype=int)
samples_to_plot = []

for i in tqdm(range(num_samples), desc="Saving samples"):
    inp = all_inputs[i]
    gt = all_gts[i]
    fname = all_meta[i]

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    vv = inp[0]
    vh = inp[1]
    sar_rgb = np.stack([np.clip(vv, 0, 1), np.clip(vh, 0, 1), np.clip(vv-vh, 0, 1)], axis=-1)
    axes[0, 0].imshow(sar_rgb)
    axes[0, 0].set_title("SAR (VV+VH)")
    axes[0, 0].axis('off')

    r = inp[4]
    g = inp[3]
    b = inp[2]
    rgb = np.stack([r, g, b], axis=-1)
    rgb = np.clip(rgb, 0, 1)
    axes[0, 1].imshow(rgb)
    axes[0, 1].set_title("Optical RGB")
    axes[0, 1].axis('off')

    gt_mask = gt.copy()
    gt_mask[gt_mask == 255] = 0
    axes[0, 2].imshow(gt_mask, cmap='viridis', vmin=0, vmax=1)
    axes[0, 2].set_title("Ground Truth")
    axes[0, 2].axis('off')

    model_names = list(MODEL_PATHS.keys())
    for j, m_name in enumerate(model_names):
        pred = all_preds_dict[m_name][i]
        r_idx = (j + 3) // 4
        c_idx = (j + 3) % 4

        axes[r_idx, c_idx].imshow(pred, cmap='viridis', vmin=0, vmax=1)
        axes[r_idx, c_idx].set_title(f"{m_name} Pred")
        axes[r_idx, c_idx].axis('off')

    water_patch = mpatches.Patch(color='yellow', label='Water (1)')
    bg_patch = mpatches.Patch(color='purple', label='Background (0)')
    fig.legend(handles=[water_patch, bg_patch], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize=12)

    plt.tight_layout()
    save_path = os.path.join(paths.OUTPUT_DIR, 'predictions', f"{fname.replace('.tif', '')}_preds.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if i in samples_to_show_idx:
        samples_to_plot.append(fig)
    else:
        plt.close(fig)

for fig in samples_to_plot:
    plt.figure(fig.number)
    plt.show()

In [22]:
all_regions = set(list(results['MUST-Former-SAR']['test_regions'].keys()) +
                  list(results['MUST-Former-SAR']['bolivia_regions'].keys()))
all_regions = sorted(list(all_regions))

heatmap_data = []
models = list(MODEL_PATHS.keys())

for region in all_regions:
    row = []
    for m in models:
        if region in results[m]['test_regions']:
            row.append(results[m]['test_regions'][region]['water_iou'])
        elif region in results[m]['bolivia_regions']:
            row.append(results[m]['bolivia_regions'][region]['water_iou'])
        else:
            row.append(np.nan)
    heatmap_data.append(row)

df_heatmap = pd.DataFrame(heatmap_data, columns=models, index=all_regions)

plt.figure(figsize=(12, 8))
sns.heatmap(df_heatmap, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=.5)
plt.title("Water IoU per Region across MUST-Former Models (Test + Bolivia)")
plt.xlabel("Models")
plt.ylabel("Regions")
plt.tight_layout()
plt.savefig(os.path.join(paths.OUTPUT_DIR, 'heatmaps', 'regional_water_iou_heatmap.png'))
plt.show()